# Ouro RQ2：独立、可关闭的重放诊断

读取已完成的 BF16/PTQ 基线结果，观察逐轮预测稳定性与隐藏状态误差。
不会重新自由生成，不修改原基线结果或配置。请使用本副本的 RQ2 ZIP。
`ENABLE_RQ2=False` 默认跳过第 7 节诊断；设为 True 开启。
原 MATH-500 notebook 可继续运行；等一个完整批次结束后再选择该批结果。
默认仅选 2 道题先跑通，之后可改为 20–50 道题；暂不包含 pass@k。


项目目录：`ouro-loopedlm-experiments`。新实验统一执行 `python scripts/package_for_colab.py` 打包。


## 1. 选择 GPU
运行时 → 更改运行时类型 → 原生 BF16 GPU（例如 L4 / A100）。

In [ ]:
import sys
import subprocess
import torch
from packaging.version import Version

if sys.version_info < (3, 10):
    raise RuntimeError("This notebook requires Python 3.10 or newer.")
if not (Version("2.6") <= Version(torch.__version__.split("+")[0]) < Version("3")):
    raise RuntimeError("Use a Colab runtime with PyTorch >= 2.6 and < 3.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a GPU runtime, then reconnect.")
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError("Native BF16 required. T4 cannot run the BF16 baseline; select L4/A100 or another BF16 GPU.")


## 2. 上传 RQ2 副本源码 ZIP
选择 `ouro-1.4b-colab-rq2-<hash>.zip`，不要覆盖原基线 notebook。


In [ ]:
from google.colab import files
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one project ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("Upload the ZIP created by scripts/package_for_colab.py.")
workspace = Path(os.environ.get("OURO_COLAB_WORKSPACE", "/content"))
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = archive.infolist()
    if len({m.filename for m in members}) != len(members):
        raise ValueError("Duplicate ZIP paths.")
    if sum(m.file_size for m in members) > 100 * 1024 * 1024:
        raise ValueError("Source bundle is unexpectedly large; exclude weights and caches.")
    roots = {PurePosixPath(m.filename).parts[0] for m in members if PurePosixPath(m.filename).parts}
    if len(roots) != 1 or not roots <= {"ouro-loopedlm-experiments", "ouro-1.4b-colab"}:
        raise ValueError("Expected a unified project ZIP or a frozen historical baseline ZIP.")
    archive_root = next(iter(roots))
    for member in members:
        path = PurePosixPath(member.filename)
        if (path.is_absolute() or ".." in path.parts or "\\" in member.filename
                or not path.parts or path.parts[0] != archive_root
                or (member.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError("Unsafe ZIP path: " + member.filename)
    prefix = archive_root + "/"
    manifest_name = prefix + "bundle_manifest.json"
    manifest = json.loads(archive.read(manifest_name))
    if manifest.get("format") != 1:
        raise ValueError("Unsupported source bundle format.")
    expected = {prefix + name for name in manifest["files"]} | {manifest_name}
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != expected:
        raise ValueError("ZIP contents do not match the source manifest.")
    for relative, entry in manifest["files"].items():
        data = archive.read(prefix + relative)
        if hashlib.sha256(data).hexdigest() != entry["sha256"] or len(data) != entry["bytes"]:
            raise ValueError("SHA256 hash mismatch: " + relative)
    run_dir = Path(tempfile.mkdtemp(prefix="ouro-source-", dir=workspace))
    for member in members:
        if not member.is_dir():
            target = run_dir.joinpath(*PurePosixPath(member.filename).parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))

PROJECT_DIR = run_dir / archive_root
print("Project directory:", PROJECT_DIR)
print("Verified source files:", len(manifest["files"]))
print("modeling_ouro.py SHA256:", manifest["files"]["ouro_local/modeling_ouro.py"]["sha256"])

## 3. 安装依赖
保留 Colab 自带 CUDA PyTorch。安装与执行均使用子进程；首次或依赖变动后运行。

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT_DIR / "requirements-bench.txt")], check=True)

## 4. 挂载保存基线结果的 Drive
默认使用同一账户的 `MyDrive/ouro-experiments`。诊断输出放在单独的 `rq2-results/`。


In [ ]:
import os
from pathlib import Path
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ouro-experiments")
else:
    STORAGE_ROOT = Path("/content/ouro-experiments")
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(STORAGE_ROOT / "hf-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUTF8"] = "1"
print("Cache and results:", STORAGE_ROOT)

## 5. 先检查本地源码
仅随机小模型，不下载完整权重。

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_rq2*.py", "-v"], cwd=PROJECT_DIR, check=True)


## 6. 探针开关与输入
只读取一个已完成的基线批次目录或下载结果 ZIP，不读取正在写入的批次。
`RQ2_LIMIT` 是题数，自动保留两种精度和全部阈值的 anchor；不是生成次数。
`MAX_PROBE_TOKENS=0` 分析全部保存 token；正数只分析前缀，仅建议调试时使用。
改变参数或重复试验请换一个新的 RQ2_RUN_NAME；当前不支持诊断续跑。


In [ ]:
ENABLE_RQ2 = False
SOURCE_RESULTS = STORAGE_ROOT / "results" / "baseline-math500-100q-early-exit-batch-01"
RQ2_DATASET = "math500"
RQ2_LIMIT = 2
RQ2_SEED = 42
MAX_PROBE_TOKENS = 0
RQ2_RUN_NAME = "rq2-math500-batch01-2q-001"
SCRATCH_DIR = Path("/content/rq2-scratch")
if not isinstance(ENABLE_RQ2, bool):
    raise ValueError("ENABLE_RQ2 must be True or False")
if not RQ2_RUN_NAME or Path(RQ2_RUN_NAME).name != RQ2_RUN_NAME or RQ2_RUN_NAME in (".", ".."):
    raise ValueError("Choose a simple RQ2_RUN_NAME")
RQ2_RESULT_DIR = STORAGE_ROOT / "rq2-results" / RQ2_RUN_NAME
print("RQ2 enabled:", ENABLE_RQ2)
print("Read-only baseline input:", SOURCE_RESULTS)
print("Separate probe output:", RQ2_RESULT_DIR)


## 7. 运行可选 RQ2 诊断
先重放 BF16 并暂存参考隐藏状态，再使用原 RTN 配方量化一次，重放 PTQ 并比较。
同一 anchor 中两种精度输入完全相同的 token 和历史退出深度。
每个 token 观察满四轮后，恢复 anchor 的 KV 规则，防止额外诊断计算影响后续历史。
原精度的预测与退出深度若不能重现原记录，会明确失败。诊断时间不混入基线耗时。


In [ ]:
command = [sys.executable, str(PROJECT_DIR / "scripts/rq2_probe.py"),
           "--source-results", str(SOURCE_RESULTS), "--output-dir", str(RQ2_RESULT_DIR),
           "--dataset", RQ2_DATASET, "--limit", str(RQ2_LIMIT), "--seed", str(RQ2_SEED),
           "--max-probe-tokens", str(MAX_PROBE_TOKENS), "--scratch-dir", str(SCRATCH_DIR)]
if ENABLE_RQ2:
    command.append("--enable-rq2")
subprocess.run(command, cwd=PROJECT_DIR, check=True)


## 8. 查看 RQ2 汇总
统计以题目为独立单位：先题内平均，再计算题间平均与标准误。
每种 anchor 精度/阈值分别显示。另一个精度的 candidate depth 是共同历史下的候选值，
不是该精度自由生成的实际深度；自由生成结果仍看原 RQ1 基线。
稳定只表示 top-1 在剩余轮次不再改变，不代表答案正确。


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
if not ENABLE_RQ2:
    print("RQ2 disabled; nothing to display.")
else:
    rq2_manifest = json.loads((RQ2_RESULT_DIR / "rq2_manifest.json").read_text())
    if rq2_manifest["status"] != "complete":
        raise RuntimeError("Probe is incomplete or failed; inspect rq2_manifest.json")
    table = pd.read_csv(RQ2_RESULT_DIR / "rq2_summary.csv")
    columns = ["dataset", "execution_mode", "threshold", "anchor_arm", "examples", "token_decisions",
               "bf16_candidate_depth", "ptq_nvfp4_candidate_depth", "candidate_depth_delta_ptq_minus_bf16",
               "bf16_stable_loop", "ptq_nvfp4_stable_loop", "bf16_late_exit_rate", "ptq_nvfp4_late_exit_rate"]
    display(table[columns])
    plt.figure(figsize=(9, 4))
    for _, row in table.iterrows():
        error_columns = sorted([c for c in table if c.startswith("l2_by_loop_") and c.rsplit("_", 1)[-1].isdigit()],
                               key=lambda c: int(c.rsplit("_", 1)[-1]))
        plt.plot(range(1, len(error_columns)+1), [row[c] for c in error_columns], marker="o",
                 label=f"{row['anchor_arm']} anchor / threshold={row['threshold']}")
    plt.xlabel("Core loop (1-based)")
    plt.ylabel("Mean per-problem hidden-state L2 error")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Diagnostic wall seconds (NOT baseline latency):", rq2_manifest["diagnostic_wall_seconds"])


## 9. 查看哪些 token 多循环
下面按共同历史下 PTQ−BF16 的候选深度差排序，显示最大的 20 条。
可在 JSONL 中按 example_id、anchor_arm、threshold、token_index 查看完整轨迹。
token_index 从 0 开始，循环轮次从 1 开始。诊断续算的深层输出是反事实观察。


In [ ]:
if ENABLE_RQ2:
    rows = []
    with (RQ2_RESULT_DIR / "rq2_tokens.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            r = json.loads(line)
            rows.append({k:r[k] for k in ["example_id", "anchor_arm", "threshold", "token_index", "token_text"]} |
                {"bf16_exit":r["bf16"]["candidate_exit_depth"], "ptq_exit":r["ptq_nvfp4"]["candidate_exit_depth"],
                 "depth_delta":r["ptq_nvfp4"]["candidate_exit_depth"]-r["bf16"]["candidate_exit_depth"],
                 "bf16_stable":r["bf16"]["stable_from_loop"], "ptq_stable":r["ptq_nvfp4"]["stable_from_loop"]})
    display(pd.DataFrame(rows).sort_values("depth_delta", ascending=False).head(20))
else:
    print("RQ2 disabled.")


## 10. 下载独立诊断结果
包含逐 token 指标、摘要和来源记录；不包含临时隐藏状态张量。


In [ ]:
if ENABLE_RQ2:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(RQ2_RESULT_DIR.parent / (RQ2_RESULT_DIR.name + "-results")),
                                  "zip", root_dir=RQ2_RESULT_DIR)
    files.download(archive)
else:
    print("RQ2 disabled; no result ZIP created.")
